# 03 Feature Engineering

### Objective

This notebook transforms the cleaned dataset into a modelling dataset by creating features required for both econometric and machine learning models. The engineered features capture exchange rate dynamics, temporal patterns, and lagged relationships that may influence food price movements.

In [25]:
# import libraries
import pandas as pd
import numpy as np

## Load Processed Dataset

The cleaned dataset produced in Notebook 02 is loaded for feature engineering.

In [26]:
#load processed dataset
data = pd.read_csv(
    "../data/processed/merged_data.csv",
    parse_dates=["Date"]
)

In [27]:
#inspect the dataset
data.head()

,Date,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,CPI,ExchangeRate
0,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,56.6,11.5665
1,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,NaN,11.5665
2,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.191734,1112101,55.1,11.5665
3,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.023029,1112102,63.0,11.5665
4,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.015759,1112301,NaN,11.5665


In [28]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 17688 entries, 0 to 17687
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 17688 non-null  datetime64[us]
 1   DivisionDescription  17688 non-null  str           
 2   GroupDescription     17688 non-null  str           
 3   ClassDescription     17688 non-null  str           
 4   SubclassDescription  17688 non-null  str           
 5   Weight               17688 non-null  float64       
 6   Eight digit code     17688 non-null  int64         
 7   CPI                  14837 non-null  float64       
 8   ExchangeRate         17688 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(4)
memory usage: 1.2 MB


In [29]:
data.describe()

,Date,Weight,Eight digit code,CPI,ExchangeRate
count,17688,17688.000000,1.768800e+04,14837.000000,17688.000000
mean,2020-06-16 02:10:54.545454,0.135893,1.158262e+06,79.614653,15.524298
min,2015-01-01 00:00:00,0.010375,1.111201e+06,39.700000,11.566500
25%,2017-09-23 12:00:00,0.030092,1.124003e+06,66.900000,13.923721
50%,2020-06-16 00:00:00,0.076704,1.151201e+06,77.700000,14.986980
75%,2023-03-08 18:00:00,0.153351,1.185101e+06,94.800000,17.578536
max,2025-12-01 00:00:00,1.447038,1.290001e+06,172.100000,19.065941
std,NaN,0.193583,3.686832e+04,16.167915,2.147451


In [30]:
#check missing values
data.isna().sum()

Date                      0
DivisionDescription       0
GroupDescription          0
ClassDescription          0
SubclassDescription       0
Weight                    0
Eight digit code          0
CPI                    2851
ExchangeRate              0
dtype: int64

In [31]:
#sort dataset
data = data.sort_values(
    ["Eight digit code", "Date"]
)

data.head()

,Date,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,CPI,ExchangeRate
0,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,56.6,11.566500
134,2015-02-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,56.8,11.577440
268,2015-03-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,56.5,12.068727
402,2015-04-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,57.0,12.012553
536,2015-05-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,57.3,11.969940


In [32]:
#create time features
data["Year"] = data["Date"].dt.year
data["Month"] = data["Date"].dt.month
data["Quarter"] = data["Date"].dt.quarter

In [33]:
#exchange rate lag features
data["ExchangeRate_Lag1"] = (
    data.groupby("Eight digit code")["ExchangeRate"]
    .shift(1)
)

data["ExchangeRate_Lag2"] = (
    data.groupby("Eight digit code")["ExchangeRate"]
    .shift(3)
)

data["ExchangeRate_Lag6"] = (
    data.groupby("Eight digit code")["ExchangeRate"]
    .shift(6)
)

In [34]:
#cpi lag featues
data["CPI_Lag1"] = (
    data.groupby("Eight digit code")["CPI"]
    .shift(1)
)

data["CPI_Lag3"] = (
    data.groupby("Eight digit code")["CPI"]
    .shift(3)
)

In [35]:
#exchange rate change
data["ExchangeRate_Change"] = (
    data.groupby("Eight digit code")["ExchangeRate"]
    .pct_change()
)

In [36]:
#depreciation and appreciation variables
data["Depreciation"] = data["ExchangeRate_Change"].clip(lower=0)

data["Appreciation"] = (
    data["ExchangeRate_Change"]
    .clip(upper=0)
    .abs()
)

In [37]:
#rolling average exchange rate
data["ExchangeRate_MA3"] = (
    data.groupby("Eight digit code")["ExchangeRate"]
    .transform(lambda x: x.rolling(3).mean())
)

In [38]:
#rolling volatility
data["ExchangeRate_STD3"] = (
    data.groupby("Eight digit code")["ExchangeRate"]
    .transform(lambda x: x.rolling(3).std())
)

In [39]:
#inspect features
data.head()

,Date,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,CPI,ExchangeRate,Year,...,ExchangeRate_Lag1,ExchangeRate_Lag2,ExchangeRate_Lag6,CPI_Lag1,CPI_Lag3,ExchangeRate_Change,Depreciation,Appreciation,ExchangeRate_MA3,ExchangeRate_STD3
0,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,56.6,11.566500,2015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
134,2015-02-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,56.8,11.577440,2015,...,11.566500,NaN,NaN,56.6,NaN,0.000946,0.000946,0.000000,NaN,NaN
268,2015-03-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,56.5,12.068727,2015,...,11.577440,NaN,NaN,56.8,NaN,0.042435,0.042435,0.000000,11.737556,0.286855
402,2015-04-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,57.0,12.012553,2015,...,12.068727,11.56650,NaN,56.5,56.6,-0.004655,0.000000,0.004655,11.886240,0.268900
536,2015-05-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,57.3,11.969940,2015,...,12.012553,11.57744,NaN,57.0,56.8,-0.003547,0.000000,0.003547,12.017073,0.049549


In [40]:
data.info()

<class 'pandas.DataFrame'>
Index: 17688 entries, 0 to 17687
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 17688 non-null  datetime64[us]
 1   DivisionDescription  17688 non-null  str           
 2   GroupDescription     17688 non-null  str           
 3   ClassDescription     17688 non-null  str           
 4   SubclassDescription  17688 non-null  str           
 5   Weight               17688 non-null  float64       
 6   Eight digit code     17688 non-null  int64         
 7   CPI                  14837 non-null  float64       
 8   ExchangeRate         17688 non-null  float64       
 9   Year                 17688 non-null  int32         
 10  Month                17688 non-null  int32         
 11  Quarter              17688 non-null  int32         
 12  ExchangeRate_Lag1    17554 non-null  float64       
 13  ExchangeRate_Lag2    17286 non-null  float64   

In [41]:
data[data["CPI"].isnull()].head()

,Date,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,CPI,ExchangeRate,Year,...,ExchangeRate_Lag1,ExchangeRate_Lag2,ExchangeRate_Lag6,CPI_Lag1,CPI_Lag3,ExchangeRate_Change,Depreciation,Appreciation,ExchangeRate_MA3,ExchangeRate_STD3
1,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,NaN,11.566500,2015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
135,2015-02-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,NaN,11.577440,2015,...,11.566500,NaN,NaN,NaN,NaN,0.000946,0.000946,0.000000,NaN,NaN
269,2015-03-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,NaN,12.068727,2015,...,11.577440,NaN,NaN,NaN,NaN,0.042435,0.042435,0.000000,11.737556,0.286855
403,2015-04-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,NaN,12.012553,2015,...,12.068727,11.56650,NaN,NaN,NaN,-0.004655,0.000000,0.004655,11.886240,0.268900
537,2015-05-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,NaN,11.969940,2015,...,12.012553,11.57744,NaN,NaN,NaN,-0.003547,0.000000,0.003547,12.017073,0.049549


In [42]:
data.groupby(
    "SubclassDescription"
)["CPI"].apply(
    lambda x: x.isnull().mean()
)

SubclassDescription
Baby food                                                                 0.333333
Bread and bakery products                                                 0.051948
Breakfast cereals                                                         0.090909
Butter and other fats and oils derived from milk                          0.901515
Cereals                                                                   0.450758
Cheese                                                                    0.180303
Chocolate, cocoa, and cocoa-based food products                           0.000000
Coffee and coffee substitutes                                             0.318182
Dates, figs and tropical fruits, fresh                                    0.000000
Eggs                                                                      0.000000
Fish                                                                      0.000000
Fish preparations                                                  

In [43]:
#encode food categories
data["SubclassDescription"].nunique()

49

In [44]:
data["SubclassDescription"].unique()

<StringArray>
[                                                               'Cereals',
                                                       'Flour of cereals',
                                              'Bread and bakery products',
                                                      'Breakfast cereals',
                 'Macaroni, noodles, couscous and similar pasta products',
                                         'Meat, fresh, chilled or frozen',
                               'Meat , dried, salted, in brine or smoked',
                    'Offal, blood and other parts of slaughtered animals',
 'Meat, offal, blood and other parts of slaughtered animals preparations',
                                                                   'Fish',
                                                      'Fish preparations',
                                                          'Other seafood',
                                                                   'Milk',
           

In [45]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

data["Food_Category"] = label_encoder.fit_transform(
    data["SubclassDescription"]
)

data[["SubclassDescription", "Food_Category"]].head()

,SubclassDescription,Food_Category
0,Cereals,4
134,Cereals,4
268,Cereals,4
402,Cereals,4
536,Cereals,4


In [46]:
#define the modelling dataset
model_data = data.dropna().copy()

In [47]:
model_data.shape

(14150, 23)

In [48]:
model_data.columns

Index(['Date', 'DivisionDescription', 'GroupDescription', 'ClassDescription',
       'SubclassDescription', 'Weight', 'Eight digit code', 'CPI',
       'ExchangeRate', 'Year', 'Month', 'Quarter', 'ExchangeRate_Lag1',
       'ExchangeRate_Lag2', 'ExchangeRate_Lag6', 'CPI_Lag1', 'CPI_Lag3',
       'ExchangeRate_Change', 'Depreciation', 'Appreciation',
       'ExchangeRate_MA3', 'ExchangeRate_STD3', 'Food_Category'],
      dtype='str')

In [49]:
model_data.isnull().sum()

Date                   0
DivisionDescription    0
GroupDescription       0
ClassDescription       0
SubclassDescription    0
Weight                 0
Eight digit code       0
CPI                    0
ExchangeRate           0
Year                   0
Month                  0
Quarter                0
ExchangeRate_Lag1      0
ExchangeRate_Lag2      0
ExchangeRate_Lag6      0
CPI_Lag1               0
CPI_Lag3               0
ExchangeRate_Change    0
Depreciation           0
Appreciation           0
ExchangeRate_MA3       0
ExchangeRate_STD3      0
Food_Category          0
dtype: int64

In [50]:
#save feature-engineered dataset
model_data.to_csv(
    "../data/processed/featured_data.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!
